In [1]:
import os

In [2]:
%pwd

'/Users/ntchindagiscard/Documents/end-end-mlflow/research'

In [3]:
os.chdir("../")

In [4]:
%pwd

'/Users/ntchindagiscard/Documents/end-end-mlflow'

In [38]:
from dataclasses import dataclass
from pathlib import Path

@dataclass(frozen=True)
class ModelTrainerConfig:
    learning_rate: float
    model_name: str
    validation_split: str
    batch_size: int
    epochs: int

In [39]:
from mlProject.utils.common import read_yaml, create_directories
from mlProject.constants import *
from mlProject import logger

In [40]:
class ConfigurationManager:
    def __init__(
            self,
            config_filepath = CONFIG_FILE_PATH,
            params_filepath = PARAMS_FILE_PATH,
            schema_filepath = SCHEMA_FILE_PATH
            ) -> None:
        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)
        self.schema = read_yaml(schema_filepath)

        create_directories([self.config.artifacts_root])

    def get_model_trainer_config(self) -> ModelTrainerConfig:

        config  = self.config.model_trainer

        create_directories([config.root_dir])

        model_trainer_config = ModelTrainerConfig(
            learning_rate = config.learning_rate,
            validation_split = config.validation_split,
            batch_size = config.batch_size,
            epochs = config.epochs,
            model_name = config.model_name
        )

        return model_trainer_config

In [8]:
# model

from tensorflow.keras import layers, Model
import tensorflow as tf
import numpy as np
from typing import Tuple, List

2025-01-19 08:48:19.204137: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


In [41]:
# model

class RecommenderNet(Model):
    def __init__(self, user_shape: int, movie_shape: int):
        super(RecommenderNet, self).__init__()
        
        # User tower layers
        self.user_input = tf.keras.layers.Input(shape=(user_shape,), name="user_input")
        self.user_NN = tf.keras.models.Sequential([
            tf.keras.layers.Dense(64, activation='relu'),
            tf.keras.layers.Dense(32, activation='relu'),
            tf.keras.layers.Dense(16)
        ])
        
        # Movie tower layers
        self.movie_input = tf.keras.layers.Input(shape=(movie_shape,), name="movie_input")
        self.movie_NN = tf.keras.models.Sequential([
            tf.keras.layers.Dense(64, activation='relu'),
            tf.keras.layers.Dense(32, activation='relu'),
            tf.keras.layers.Dense(16)
        ])
        
        # Combined layers
        self.output_layer = tf.keras.layers.Dot(axes=1)
        
    def call(self, inputs):
        user_input, movie_input = inputs
        
        # User tower
        vu = self.user_NN(user_input)
        vu = tf.linalg.l2_normalize(vu, axis=1)
        
        # Movie tower
        vm = self.movie_NN(movie_input)
        vm = tf.linalg.l2_normalize(vm, axis=1)
        
        return self.output_layer([vu, vm])
    
    def build_graph(self):
        """Create model graph for visualization"""
        model = Model(
            inputs=[self.user_input, self.movie_input],
            outputs=self.call([self.user_input, self.movie_input])
        )
        logger.info(f"Model summary: {model.summary()}")
        return model

In [61]:
class RecommenderTrainer:
    def __init__(
        self,
        config: ModelTrainerConfig,
        user_shape: int,
        movie_shape: int,
        # learning_rate: float = 0.001,
        # batch_size: int = 32,
        # epochs: int = 10
    ):
        self.model = RecommenderNet(user_shape, movie_shape)
        self.learning_rate = config.learning_rate
        self.batch_size = config.batch_size
        self.epochs = config.epochs
        self.history = None
        self.cost_fn = tf.keras.losses.MeanSquaredError()
        self.validation_split = config.validation_split
        
    def compile_model(self):
        """Compile the model with specified parameters"""
        optimizer = tf.keras.optimizers.Adam(learning_rate=self.learning_rate)
        self.model.compile(
            optimizer=optimizer,
            loss= self.cost_fn,
            metrics=['mae', 'mse']
        )
        
    def train(
        self,
        X_user: np.ndarray,
        X_movie: np.ndarray,
        y: np.ndarray,
        callbacks: List = None
    ):
        """Train the model"""
        print(f"Validation split {self.validation_split}")
        validation_split = self.validation_split,
        self.compile_model()
        self.history = self.model.fit(
            [X_user, X_movie],
            y,
            batch_size=self.batch_size,
            epochs=self.epochs,
            validation_split=self.validation_split,
            callbacks=callbacks
        )
        return self.history
    
    def predict(self, X_user: np.ndarray, X_movie: np.ndarray) -> np.ndarray:
        """Make predictions"""
        return self.model.predict([X_user, X_movie])
    
    def evaluate(
        self,
        X_user: np.ndarray,
        X_movie: np.ndarray,
        y: np.ndarray
    ) -> Tuple[float, float]:
        """Evaluate the model"""
        return self.model.evaluate([X_user, X_movie], y)
    
    def save_model(self, path: str):
        """Save the model"""
        self.model.save(path)
    
    @staticmethod
    def load_model(path: str):
        """Load a saved model"""
        return tf.keras.models.load_model(path)


In [44]:
import numpy as np

# Define dimensions
num_samples = 10  # Number of samples
user_vector_size = 21  # Length of each user vector
movie_vector_size = 71  # Length of each movie vector
target_size = 1  # Target output size

# Generate random data
X_user = np.random.rand(num_samples, user_vector_size)  # Shape (10, 21)
X_movie = np.random.rand(num_samples, movie_vector_size)  # Shape (10, 71)
y = np.random.rand(num_samples)  # Shape (10, 1)

In [45]:
y

array([0.7277152 , 0.56312436, 0.26276974, 0.50456946, 0.09121046,
       0.5755808 , 0.41245455, 0.28075273, 0.90899189, 0.58039716])

In [62]:
try:
    config = ConfigurationManager()
    model_trainer_config = config.get_model_trainer_config()
    model_trainer = RecommenderTrainer(config=model_trainer_config, user_shape=user_vector_size, movie_shape=movie_vector_size)
    callbacks = [
        tf.keras.callbacks.EarlyStopping(
            monitor='val_loss',
            patience=3,
            restore_best_weights=True
        )
    ]
    history = model_trainer.train(
        X_user,
        X_movie,
        y,
        callbacks=callbacks
    )
except Exception as e:
    logger.exception(f"Oops😟! An error occured: {e} ")

[2025-01-19 15:34:17,623: INFO: common: Yaml file : config/config.yaml loaded successfully]
[2025-01-19 15:34:17,658: INFO: common: Yaml file : params.yaml loaded successfully]
[2025-01-19 15:34:17,738: INFO: common: Yaml file : schema.yaml loaded successfully]
[2025-01-19 15:34:17,746: INFO: common: Created directory at: artifacts]
[2025-01-19 15:34:17,760: INFO: common: Created directory at: artifacts/model_trainer]
Validation split 0.2


1/1 ━━━━━━━━━━━━━━━━━━━━ 27s 27s/step - loss: 0.3480 - mae: 0.5233 - mse: 0.3480 - val_loss: 0.8496 - val_mae: 0.8805 - val_mse: 0.8496


In [ ]:
class ModelTrainerPipeline:

    def __init__(self) -> None:
        pass

    def main()